In [19]:
import os
import sys
import re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.cluster import KMeans
import torch

PATH = "../.."
# PATH = "storage"


github_path = os.path.join(PATH, "third_party", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer

# HuggingFace login
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

# For transforms
from torchvision import transforms
import timm
from timm.data import resolve_data_config, create_transform
from transformers import AutoImageProcessor, AutoModel, AutoModelForImageClassification

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [20]:
class ProcessorTransform:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, img):
        # Return only pixel_values (tensor)
        return self.processor(img, return_tensors="pt")["pixel_values"].squeeze(0)

# Torch/timm models
uni_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
giga_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
exaone_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
hoptimus_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517))
])
virchow_transform = transforms.Compose([
    transforms.Resize(size=224, interpolation=transforms.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4850, 0.4560, 0.4060), std=(0.2290, 0.2240, 0.2250))
])

# HuggingFace processor models
resnet_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
hibou_b_processor   = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True)
hibou_L_processor   = AutoImageProcessor.from_pretrained("histai/hibou-L", trust_remote_code=True)
phikon_processor    = AutoImageProcessor.from_pretrained("owkin/phikon", trust_remote_code=True)
phikonv2_processor  = AutoImageProcessor.from_pretrained("owkin/phikon-v2", trust_remote_code=True)

resnet_transform = ProcessorTransform(resnet_processor)
hibou_b_transform  = ProcessorTransform(hibou_b_processor)
hibou_L_transform  = ProcessorTransform(hibou_L_processor)
phikon_transform   = ProcessorTransform(phikon_processor)
phikonv2_transform = ProcessorTransform(phikonv2_processor)

model_transforms = {
    "ResNet18":     resnet_transform,
    "ResNet50":     resnet_transform,
    "UNI":          uni_transform,
    "UNI2":         uni_transform,
    "GigaPath":     giga_transform,
    "Virchow":      virchow_transform,
    "Virchow2":     virchow_transform,
    "EXAONEPath":   exaone_transform,
    "Hibou-B":      hibou_b_transform,
    "Hibou-L":      hibou_L_transform,
    "H-Optimus-0":  hoptimus_transform,
    "H-Optimus-1":  hoptimus_transform,
    "H0-mini":      hoptimus_transform,
    "Phikon v1":    phikon_transform,
    "Phikon v2":    phikonv2_transform,
}

def timm_transform(model):
    # Check if model has pretrained_cfg directly, else check .backbone
    if hasattr(model, "pretrained_cfg"):
        timm_model = model
    elif hasattr(model, "backbone") and hasattr(model.backbone, "pretrained_cfg"):
        timm_model = model.backbone
    else:
        raise AttributeError("Cannot find .pretrained_cfg on the model or its backbone.")
    cfg = resolve_data_config(timm_model.pretrained_cfg, model=timm_model)
    return create_transform(**cfg)

def get_transform(model_name, model=None):
    if model_name in {"Virchow", "Virchow2", "H0-mini"} and model is not None:
        return timm_transform(model)
    return model_transforms[model_name]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [21]:
model_configs = [
    {
        "name": "ResNet18",
        "enabled": True,
        "init": lambda: AutoModelForImageClassification.from_pretrained("microsoft/resnet-18").eval().to("cuda"),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).logits.detach().cpu().numpy().squeeze(),
    },
    {
        "name": "ResNet50",
        "enabled": True,
        "init": lambda: timm.create_model("resnet50", pretrained=True).eval().to("cuda"),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "UNI",
        "enabled": True,
        "init": lambda: timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True).eval().to("cuda"),
        "transform": uni_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "UNI2",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:MahmoodLab/UNI2-h",
            pretrained=True,
            img_size=224,
            patch_size=14,
            depth=24,
            num_heads=24,
            init_values=1e-5,
            embed_dim=1536,
            mlp_ratio=2.66667*2,
            num_classes=0,
            no_embed_class=True,
            mlp_layer=timm.layers.SwiGLUPacked,
            act_layer=torch.nn.SiLU,
            reg_tokens=8,
            dynamic_img_size=True,
        ).eval().to("cuda"),
        "transform": uni_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "GigaPath",
        "enabled": True,
        "init": lambda: timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True).eval().to("cuda"),
        "transform": giga_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Virchow",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:paige-ai/Virchow",
            pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked,
            act_layer=torch.nn.SiLU,
        ).eval().to("cuda"),
        "transform": virchow_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to("cuda"))[:, 0],  # class token
            model(tensor.to("cuda"))[:, 1:].mean(1)
        ], dim=-1).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Virchow2",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:paige-ai/Virchow2",
            pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked,
            act_layer=torch.nn.SiLU,
        ).eval().to("cuda"),
        "transform": virchow_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to("cuda"))[:, 0],  # class token
            model(tensor.to("cuda"))[:, 5:].mean(1)  # patch tokens (ignore 1-4)
        ], dim=-1).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "EXAONEPath",
        "enabled": True,
        "init": lambda: VisionTransformer.from_pretrained(
            "LGAI-EXAONE/EXAONEPath", use_auth_token=hf_token
        ).eval().cuda(),
        "transform": exaone_transform,
        "embedding_fn": lambda model, tensor: model(tensor.cuda()).squeeze().cpu().numpy(),
    },
    {
        "name": "Hibou-B",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("histai/hibou-b", trust_remote_code=True).eval().to("cuda"),
        "transform": hibou_b_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Hibou-L",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("histai/hibou-L", trust_remote_code=True).eval().to("cuda"),
        "transform": hibou_L_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Phikon v1",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("owkin/phikon", trust_remote_code=True).eval().to("cuda"),
        "transform": phikon_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Phikon v2",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("owkin/phikon-v2", trust_remote_code=True).eval().to("cuda"),
        "transform": phikonv2_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "H-Optimus-0",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H-optimus-0", pretrained=True, init_values=1e-5, dynamic_img_size=False
        ).eval().to("cuda"),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "H-Optimus-1",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H-optimus-1", pretrained=True, init_values=1e-5, dynamic_img_size=False
        ).eval().to("cuda"),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "H0-mini",
        "enabled": False,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H0-mini", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU
        ).eval().to("cuda"),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
]

In [22]:
# PARAMETERS
image_dir = os.path.join(PATH, "data", "PatchCamelyon", "train_images_sampled")
n_clusters = 5

# Pattern explanation:
# (.+?_\d+)       # non-greedy: everything up to (and including) the second underscore
# _(\d+)          # image index (but not needed)
# _(TRUE|FALSE)   # class
# \.png$          # file ending
pattern = re.compile(r"^(.+?_\d+)_\d+_(TRUE|FALSE)\.png$")

wsi_to_paths = {}
for fname in os.listdir(image_dir):
    match = pattern.match(fname)
    if not match:
        continue
    wsi = match.group(1)  # e.g., "tumor_002"
    wsi_to_paths.setdefault(wsi, []).append(os.path.join(image_dir, fname))

# (Optional) Check
print("Number of WSIs:", len(wsi_to_paths))
for k in list(wsi_to_paths.keys())[:5]:
    print(k, ":", len(wsi_to_paths[k]), "patches")

Number of WSIs: 46
tumor_002 : 380 patches
tumor_004 : 348 patches
tumor_007 : 358 patches
tumor_009 : 334 patches
tumor_016 : 334 patches


In [23]:
import pickle
import os
import random

save_base = os.path.join(PATH, "data", "PatchCamelyon")

# Find the minimum patch count across all WSIs (for balanced sampling)
min_patches = min(len(paths) for paths in wsi_to_paths.values())
max_patches_per_wsi = min_patches  # Or set to a fixed value, e.g. 300

random.seed(42)  # For reproducibility

for config in model_configs:
    if not config["enabled"]:
        continue

    emb_dir = os.path.join(save_base, "embeddings")
    os.makedirs(emb_dir, exist_ok=True)
    pkl_path = os.path.join(emb_dir, f"{config['name']}_patch_embeddings.pkl")
    if os.path.exists(pkl_path):
        print(f"Patch embedding file already exists for {config['name']} – skipping.")
        continue

    print(f"\n=== Extracting and saving patch embeddings for: {config['name']} ===")
    model = config["init"]()
    transform = config["transform"]

    patch_infos = []  # To hold all patch-level data

    for wsi, paths in tqdm(wsi_to_paths.items(), desc=f"{config['name']} patch embeddings"):
        # BALANCE: limit patches per WSI
        if len(paths) > max_patches_per_wsi:
            paths = random.sample(paths, max_patches_per_wsi)
        # Proceed as before
        for img_path in paths:
            img = Image.open(img_path).convert("RGB")
            tensor = transform(img).unsqueeze(0)
            with torch.no_grad():
                emb = config["embedding_fn"](model, tensor)
            patch_infos.append({
                "wsi": wsi,
                "img_path": img_path,
                "embedding": emb  # numpy array
            })

    with open(pkl_path, "wb") as f:
        pickle.dump(patch_infos, f)
    print(f"Saved patch embeddings: {pkl_path}")

Patch embedding file already exists for ResNet18 – skipping.
Patch embedding file already exists for ResNet50 – skipping.
Patch embedding file already exists for UNI – skipping.
Patch embedding file already exists for UNI2 – skipping.
Patch embedding file already exists for GigaPath – skipping.
Patch embedding file already exists for Virchow – skipping.
Patch embedding file already exists for Virchow2 – skipping.
Patch embedding file already exists for EXAONEPath – skipping.
Patch embedding file already exists for Hibou-B – skipping.
Patch embedding file already exists for Hibou-L – skipping.
Patch embedding file already exists for Phikon v1 – skipping.
Patch embedding file already exists for Phikon v2 – skipping.
Patch embedding file already exists for H-Optimus-0 – skipping.
Patch embedding file already exists for H-Optimus-1 – skipping.


## Balanced K-means

In [24]:
# import pickle
# import os
# import pandas as pd
# from k_means_constrained import KMeansConstrained  # Note: different from sklearn!
# # ...rest of your imports as before

# save_base = os.path.join(PATH, "data", "PatchCamelyon")

# for config in model_configs:
#     if not config["enabled"]:
#         continue

#     emb_dir = os.path.join(save_base, "embeddings")
#     pkl_path = os.path.join(emb_dir, f"{config['name']}_patch_embeddings.pkl")

#     with open(pkl_path, "rb") as f:
#         patch_infos = pickle.load(f)

#     # 1. Compute mean embedding per WSI
#     wsi_embeddings = {}
#     for wsi in wsi_to_paths.keys():
#         embeds = [p["embedding"] for p in patch_infos if p["wsi"] == wsi]
#         wsi_embeddings[wsi] = np.mean(np.stack(embeds), axis=0)

#     # 2. Balanced K-means clustering on WSI embeddings
#     wsis = list(wsi_embeddings.keys())
#     X = np.stack([wsi_embeddings[wsi] for wsi in wsis])
#     n_wsis = len(wsis)
#     cluster_size = n_wsis // n_clusters  # e.g. for 50 WSIs, 5 clusters = 10 per cluster

#     clf = KMeansConstrained(
#         n_clusters=n_clusters,
#         size_min=cluster_size,
#         size_max=cluster_size+1,  # Allow at most 1 more in some clusters if not perfectly divisible
#         random_state=42
#     )
#     clusters = clf.fit_predict(X)

#     # 3. Output CSV
#     cluster_dir = os.path.join(save_base, "clusters")
#     os.makedirs(cluster_dir, exist_ok=True)
#     out_csv = os.path.join(cluster_dir, f"kmeans_folds_{config['name']}.csv")
#     pd.DataFrame({"wsi": wsis, "fold": clusters}).to_csv(out_csv, index=False)
#     print(f"Saved: {out_csv}")

## K-Means

### WSI

In [28]:
import pickle
import os

save_base = os.path.join(PATH, "data", "PatchCamelyon")

for config in model_configs:
    if not config["enabled"]:
        continue

    emb_dir = os.path.join(save_base, "embeddings")
    pkl_path = os.path.join(emb_dir, f"{config['name']}_patch_embeddings.pkl")

    with open(pkl_path, "rb") as f:
        patch_infos = pickle.load(f)

    # 1. Compute mean embedding per WSI
    wsi_embeddings = {}
    for wsi in wsi_to_paths.keys():
        embeds = [p["embedding"] for p in patch_infos if p["wsi"] == wsi]
        wsi_embeddings[wsi] = np.mean(np.stack(embeds), axis=0)

    # 2. K-means clustering on WSI embeddings
    wsis = list(wsi_embeddings.keys())
    X = np.stack([wsi_embeddings[wsi] for wsi in wsis])
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X)

    # 3. Output CSV
    cluster_dir = os.path.join(save_base, "clusters")
    os.makedirs(cluster_dir, exist_ok=True)
    out_csv = os.path.join(cluster_dir, f"kmeans_folds_{config['name']}.csv")
    pd.DataFrame({"wsi": wsis, "fold": clusters}).to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

Saved: data\PatchCamelyon\clusters\kmeans_folds_ResNet18.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_ResNet50.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_UNI.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_UNI2.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_GigaPath.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_Virchow.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_Virchow2.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_EXAONEPath.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_Hibou-B.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_Hibou-L.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_Phikon v1.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_Phikon v2.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_H-Optimus-0.csv
Saved: data\PatchCamelyon\clusters\kmeans_folds_H-Optimus-1.csv


### Patch (coloured by cluster)

In [26]:
import pickle
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

save_base = os.path.join(PATH, "data", "PatchCamelyon")
n_clusters = 5  # Set to your desired number of patch clusters
tsne_perplexity = 50
tsne_random_state = 42

for config in model_configs:
    if not config["enabled"]:
        continue

    print(f"\nProcessing {config['name']} patch-level clustering...")

    emb_dir = os.path.join(save_base, "embeddings")
    pkl_path = os.path.join(emb_dir, f"{config['name']}_patch_embeddings.pkl")

    with open(pkl_path, "rb") as f:
        patch_infos = pickle.load(f)

    # 1. Gather all patch embeddings and their metadata
    patch_embeds = np.stack([p["embedding"] for p in patch_infos])
    wsis = [p["wsi"] for p in patch_infos]
    img_paths = [p.get("img_path", "") for p in patch_infos]

    # 2. KMeans clustering on patch embeddings
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    patch_clusters = kmeans.fit_predict(patch_embeds)

    # 3. Output CSV with patch cluster assignments
    cluster_dir = os.path.join(save_base, "clusters")
    os.makedirs(cluster_dir, exist_ok=True)
    out_csv = os.path.join(cluster_dir, f"{config['name']}_patch_kmeans.csv")
    df_out = pd.DataFrame({
        "wsi": wsis,
        "img_path": img_paths,
        "patch_cluster": patch_clusters,
    })
    df_out.to_csv(out_csv, index=False)
    print(f"Saved patch cluster assignments: {out_csv}")

    # 4. Optional: t-SNE visualization
    print(f"Running t-SNE for {config['name']} ...")
    tsne = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=tsne_random_state)
    patch_embeds_2d = tsne.fit_transform(patch_embeds)

    plt.figure(figsize=(10, 8))
    for c in range(n_clusters):
        idxs = np.where(patch_clusters == c)[0]
        plt.scatter(
            patch_embeds_2d[idxs, 0],
            patch_embeds_2d[idxs, 1],
            s=5, alpha=0.7, label=f"Cluster {c} (n={len(idxs)})"
        )
    plt.title(f"{config['name']} Patch-level KMeans Clusters (t-SNE)")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(markerscale=3, fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    tsne_dir = os.path.join(save_base, "tSNE")
    os.makedirs(tsne_dir, exist_ok=True)
    plot_path = os.path.join(tsne_dir, f"{config['name']}_patch_kmeans_tsne.png")
    plt.savefig(plot_path, dpi=200)
    plt.close()
    print(f"Saved t-SNE plot: {plot_path}")


Processing ResNet18 patch-level clustering...
Saved patch cluster assignments: data\PatchCamelyon\clusters\ResNet18_patch_kmeans.csv
Running t-SNE for ResNet18 ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\ResNet18_patch_kmeans_tsne.png

Processing ResNet50 patch-level clustering...
Saved patch cluster assignments: data\PatchCamelyon\clusters\ResNet50_patch_kmeans.csv
Running t-SNE for ResNet50 ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\ResNet50_patch_kmeans_tsne.png

Processing UNI patch-level clustering...
Saved patch cluster assignments: data\PatchCamelyon\clusters\UNI_patch_kmeans.csv
Running t-SNE for UNI ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\UNI_patch_kmeans_tsne.png

Processing UNI2 patch-level clustering...
Saved patch cluster assignments: data\PatchCamelyon\clusters\UNI2_patch_kmeans.csv
Running t-SNE for UNI2 ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\UNI2_patch_kmeans_tsne.png

Processing GigaPath patch-level clustering...
Saved patch cluster assignments: data

## Patch (Coloured by class)

In [27]:
import pickle
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

save_base = os.path.join(PATH, "data", "PatchCamelyon")
n_clusters = 5  # or any number you use
tsne_perplexity = 50
tsne_random_state = 42

for config in model_configs:
    if not config["enabled"]:
        continue

    print(f"\nProcessing {config['name']} patch-level clustering...")

    emb_dir = os.path.join(save_base, "embeddings")
    pkl_path = os.path.join(emb_dir, f"{config['name']}_patch_embeddings.pkl")

    with open(pkl_path, "rb") as f:
        patch_infos = pickle.load(f)

    # 1. Gather all patch embeddings and their metadata
    patch_embeds = np.stack([p["embedding"] for p in patch_infos])
    wsis = [p["wsi"] for p in patch_infos]
    img_paths = [p.get("img_path", "") for p in patch_infos]

    # 2. KMeans clustering on patch embeddings
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    patch_clusters = kmeans.fit_predict(patch_embeds)

    # 3. t-SNE for all patches
    print(f"Running t-SNE for {config['name']} ...")
    tsne = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=tsne_random_state)
    patch_embeds_2d = tsne.fit_transform(patch_embeds)

    # 4. Plot by patch cluster (original behavior)
    plt.figure(figsize=(10, 8))
    for c in range(n_clusters):
        idxs = np.where(patch_clusters == c)[0]
        plt.scatter(
            patch_embeds_2d[idxs, 0],
            patch_embeds_2d[idxs, 1],
            s=5, alpha=0.7, label=f"Cluster {c} (n={len(idxs)})"
        )
    plt.title(f"{config['name']} Patch-level KMeans Clusters (t-SNE)")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(markerscale=3, fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    tsne_dir = os.path.join(save_base, "tSNE")
    os.makedirs(tsne_dir, exist_ok=True)
    plot_path = os.path.join(tsne_dir, f"{config['name']}_patch_kmeans_tsne.png")
    plt.savefig(plot_path, dpi=200)
    plt.close()
    print(f"Saved t-SNE plot: {plot_path}")

    # 5. Plot by class (new feature)
    # Try to get class for each patch (TRUE/1 for tumor, FALSE/0 for normal)
    if "label" in patch_infos[0]:
        patch_classes = [p["label"] for p in patch_infos]
    elif "class" in patch_infos[0]:
        patch_classes = [p["class"] for p in patch_infos]
    elif "tumor_patch" in patch_infos[0]:
        patch_classes = [1 if p["tumor_patch"] in ["TRUE", 1, True] else 0 for p in patch_infos]
    else:
        patch_classes = [1 if str(p.get("img_path","")).upper().endswith("TRUE.PNG") else 0 for p in patch_infos]

    class_names = {0: "FALSE/Normal", 1: "TRUE/Tumor"}
    class_colors = ["tab:blue", "tab:red"]

    plt.figure(figsize=(10, 8))
    for c in [0, 1]:
        idxs = np.where(np.array(patch_classes) == c)[0]
        plt.scatter(
            patch_embeds_2d[idxs, 0],
            patch_embeds_2d[idxs, 1],
            s=5, alpha=0.6, label=class_names[c], color=class_colors[c]
        )
    plt.title(f"{config['name']} Patch t-SNE by Class")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(markerscale=3, fontsize=12)
    plt.tight_layout()
    byclass_dir = os.path.join(save_base, "tSNE", "by_class")
    os.makedirs(byclass_dir, exist_ok=True)
    plot_path = os.path.join(byclass_dir, f"{config['name']}_patch_tsne_byclass.png")
    plt.savefig(plot_path, dpi=200)
    plt.close()
    print(f"Saved t-SNE colored by class: {plot_path}")

    # 6. Output CSV with patch cluster assignments (as before)
    cluster_dir = os.path.join(save_base, "clusters")
    os.makedirs(cluster_dir, exist_ok=True)
    out_csv = os.path.join(cluster_dir, f"{config['name']}_patch_kmeans.csv")
    df_out = pd.DataFrame({
        "wsi": wsis,
        "img_path": img_paths,
        "patch_cluster": patch_clusters,
        "patch_class": patch_classes,
    })
    df_out.to_csv(out_csv, index=False)
    print(f"Saved patch cluster assignments: {out_csv}")


Processing ResNet18 patch-level clustering...
Running t-SNE for ResNet18 ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\ResNet18_patch_kmeans_tsne.png
Saved t-SNE colored by class: data\PatchCamelyon\tSNE\by_class\ResNet18_patch_tsne_byclass.png
Saved patch cluster assignments: data\PatchCamelyon\clusters\ResNet18_patch_kmeans.csv

Processing ResNet50 patch-level clustering...
Running t-SNE for ResNet50 ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\ResNet50_patch_kmeans_tsne.png
Saved t-SNE colored by class: data\PatchCamelyon\tSNE\by_class\ResNet50_patch_tsne_byclass.png
Saved patch cluster assignments: data\PatchCamelyon\clusters\ResNet50_patch_kmeans.csv

Processing UNI patch-level clustering...
Running t-SNE for UNI ...
Saved t-SNE plot: data\PatchCamelyon\tSNE\UNI_patch_kmeans_tsne.png
Saved t-SNE colored by class: data\PatchCamelyon\tSNE\by_class\UNI_patch_tsne_byclass.png
Saved patch cluster assignments: data\PatchCamelyon\clusters\UNI_patch_kmeans.csv

Processing UNI2 patch-lev

## Plot T-SNE

In [29]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import pickle
import numpy as np

save_base = os.path.join(PATH, "data", "PatchCamelyon")
tsne_perplexity = 120
tsne_random_state = 42

for config in model_configs:
    if not config["enabled"]:
        continue

    emb_dir = os.path.join(save_base, "embeddings")
    cluster_dir = os.path.join(save_base, "clusters")
    tsne_dir = os.path.join(save_base, "tSNE")

    emb_file = os.path.join(emb_dir, f"{config['name']}_patch_embeddings.pkl")
    out_csv = os.path.join(cluster_dir, f"kmeans_folds_{config['name']}.csv")
    if not (os.path.exists(emb_file) and os.path.exists(out_csv)):
        print(f"Missing data for {config['name']}, skipping plot.")
        continue

    with open(emb_file, "rb") as f:
        patch_infos = pickle.load(f)
    df = pd.read_csv(out_csv)
    wsi_to_cluster = dict(zip(df["wsi"], df["fold"]))

    patch_embeds = []
    patch_cluster = []
    for p in patch_infos:
        wsi = p["wsi"]
        if wsi not in wsi_to_cluster:
            continue
        patch_embeds.append(p["embedding"])
        patch_cluster.append(wsi_to_cluster[wsi])

    patch_embeds = np.stack(patch_embeds)
    patch_cluster = np.array(patch_cluster)

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=tsne_random_state)
    patch_embeds_2d = tsne.fit_transform(patch_embeds)

    # Count images per cluster
    unique_clusters, counts = np.unique(patch_cluster, return_counts=True)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_clusters)))
    plt.figure(figsize=(9, 7))

    for idx, (c, count, color) in enumerate(zip(unique_clusters, counts, colors)):
        mask = (patch_cluster == c)
        plt.scatter(
            patch_embeds_2d[mask, 0], patch_embeds_2d[mask, 1],
            color=color, s=5, alpha=0.6, label=f"Cluster {c} (N={count})"
        )

    plt.title(f"{config['name']} patch embeddings (t-SNE, colored by WSI cluster)")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(markerscale=3, fontsize=10)
    plt.tight_layout()
    os.makedirs(tsne_dir, exist_ok=True)
    tsne_path = os.path.join(tsne_dir, f"{config['name']}_patch_tsne_by_wsi_cluster.png")
    plt.savefig(tsne_path, dpi=200)
    plt.close()
    print(f"Saved patch-level t-SNE plot: {tsne_path}")

Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\ResNet18_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\ResNet50_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\UNI_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\UNI2_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\GigaPath_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\Virchow_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\Virchow2_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\EXAONEPath_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\Hibou-B_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\Hibou-L_patch_tsne_by_wsi_cluster.png
Saved patch-level t-SNE plot: data\PatchCamelyon\tSNE\Phikon v1_patch_tsne_by_ws